# Lab 1 · Catch the fraud — Supervised Learning

**Fundamentals of AI** · [Dataset (Kaggle/ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud): 284,807 card transactions, 492 fraud.

## 1 · Setup & load

**Install libraries** (safe to skip on Colab).

In [ ]:
#@title 0. Install libraries (Colab-ready) { display-mode: "form" }
# Colab already has these; this makes the notebook work anywhere. -q = quiet.
%pip install -q scikit-learn pandas matplotlib ipywidgets

**Imports + helpers.** `plot_cm` draws a dark confusion matrix; `need()` stops the notebook politely if a step was skipped.

In [ ]:
#@title 1a. Theme + plot_cm() helper { display-mode: "form" }
%matplotlib inline   # charts show inside the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix

# dark theme colors used by all charts in this notebook
BG, ACCENT, ACCENT2, TEXT = "#030503", "#00E676", "#3DFF8A", "#FFFFFF"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": TEXT, "axes.labelcolor": TEXT, "axes.edgecolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "axes.titlecolor": TEXT,
    "grid.color": "#1c3a24", "font.size": 14, "axes.titlesize": 18,
    "axes.labelsize": 15, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "legend.facecolor": "#0a140d", "legend.edgecolor": ACCENT,
    "figure.figsize": (8, 5),
})

def plot_cm(y_true, y_pred, title=""):
    """Dark-themed confusion matrix. Rows = truth, cols = prediction."""
    cm = confusion_matrix(y_true, y_pred)   # 2x2: [[TN, FP], [FN, TP]]
    fig, ax = plt.subplots()
    ax.imshow(cm, cmap="Greens")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["legit", "fraud"], color=TEXT)   # predicted
    ax.set_yticklabels(["legit", "fraud"], color=TEXT)   # actual
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actually")
    ax.set_title(title)
    for i in range(2):                      # print the count inside each cell
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                    color=TEXT, fontsize=18, fontweight="bold")
    plt.tight_layout(); plt.show()

def need(*names):
    """Friendly stop if an earlier step hasn't run yet."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise SystemExit(f"✏️ Not ready yet — missing: {', '.join(missing)}. Run the earlier cells first.")
print("Theme ready.")

**Load the data** — local CSV first, OpenML fallback, clear instructions if both fail.

In [ ]:
#@title 1b. Load the data { display-mode: "form" }
import os

# look for a local CSV first (data/creditcard.csv, parent dir, or notebook dir)
csv_path = next((p for p in ["data/creditcard.csv", "../data/creditcard.csv", "creditcard.csv"]
                 if os.path.exists(p)), None)

if csv_path:
    print(f"Loading local file: {csv_path}")
    _df = pd.read_csv(csv_path)
    X_all = _df.drop(columns=["Class"])      # features: all columns except the label
    y_all = _df["Class"].astype(int).values  # labels: 0 = legit, 1 = fraud
else:
    try:
        from sklearn.datasets import fetch_openml
        # OpenML mirror of the Kaggle dataset — no login needed
        data = fetch_openml(data_id=1597, as_frame=True, parser="auto")
        X_all = data.data if hasattr(data, "data") else data[0]
        y_all = (data.target if hasattr(data, "target") else data[1]).astype(int).values
    except Exception as e:
        # download failed → explain the manual fallback and stop cleanly
        print("OpenML download failed:", e)
        print("FALLBACK: download creditcard.csv from")
        print("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  (free login),")
        print("put it in a 'data/' folder next to this notebook (or upload via the file panel),")
        print("then re-run this cell.")
        raise SystemExit("Dataset not loaded yet.")
print("Loaded OK.")

**Peek at the data.**

In [ ]:
#@title 1c. Peek at the data { display-mode: "form" }
print(f"Shape: {X_all.shape[0]:,} rows × {X_all.shape[1]} columns")   # rows = transactions, cols = features
X_all.head()  # first 5 rows — most columns (V1–V28) are anonymized PCA features

> If 1b fails: download `creditcard.csv` from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), upload it in Colab's file panel, re-run 1b.

## 2 · How rare is fraud?

> 📝 Guess the fraud % *before* running the next cells.

**Count the classes.**

In [ ]:
#@title 2a. The numbers { display-mode: "form" }
# value_counts() tallies how many rows have each label (0 = legit, 1 = fraud)
counts = pd.Series(y_all).value_counts().sort_index()   # index 0 = legit count, 1 = fraud count
rate = counts.get(1, 0) / len(y_all) * 100          # fraud share as a percentage
print(f"Fraud: {counts.get(1,0):,}  ({rate:.3f}%)")
print(f"Legit: {counts.get(0,0):,}  ({100-rate:.3f}%)")
# how many legit transactions per single fraud (max(...,1) avoids divide-by-zero)
print(f"Ratio: ~1 fraud per {counts.get(0,0)/max(counts.get(1,0),1):,.0f} legit transactions")

**Chart the class balance** — log scale so the tiny fraud bar is visible.

In [ ]:
#@title 2b. Class balance chart { display-mode: "form" }
fig, ax = plt.subplots()   # one figure with one set of axes
# one bar per class; green = legit, red = fraud
bars = ax.bar(["Legit (0)", "Fraud (1)"], [counts.get(0,0), counts.get(1,0)],
              color=[ACCENT2, "#FF5252"], width=0.5)
ax.set_yscale("log")   # log scale — otherwise the fraud bar is invisible (100x smaller)
ax.set_ylabel("Transactions (log scale)")
ax.set_title("Class balance")
# loop over the 2 bars and print the exact count just above each one
for b, v in zip(bars, [counts.get(0,0), counts.get(1,0)]):
    ax.text(b.get_x()+b.get_width()/2, v*1.2, f"{v:,}", ha="center", fontsize=14)
plt.tight_layout()   # adjust spacing so labels fit
plt.show()

## 3 · The baseline trap

Model that **always says "legit"**. Guess its accuracy *before* running.

> 💡 Key idea: accuracy can look great while catching **zero** fraud.

**Train + score a model that always says "legit".**

In [ ]:
#@title 3a. Always-legit model — scores { display-mode: "form" }
from sklearn.dummy import DummyClassifier          # a "model" that ignores features
from sklearn.metrics import accuracy_score          # share of correct predictions

# strategy="most_frequent" = always predict whichever label appears most often
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_all, y_all)               # "learns" only: legit is the majority class
pred_dummy = dummy.predict(X_all)     # predicts 0 (legit) for every transaction

acc = accuracy_score(y_all, pred_dummy)   # compare predictions vs real labels
print(f"Accuracy: {acc*100:.2f}%   ← looks amazing!")
# how many real frauds did it catch? (y_all.sum() = total fraud count in data)
print(f"Frauds caught: {(pred_dummy==1).sum()} / {int(y_all.sum())}")

**Its confusion matrix.**

In [ ]:
#@title 3b. Its confusion matrix { display-mode: "form" }
# plot_cm(y_true, y_pred, title) was defined in cell 1a — reuse it here
plot_cm(y_all, pred_dummy, title="Always-legit — high accuracy, 0 frauds caught")

## 4 · Train/test split

Train on part of the data, **test on data the model never saw**.

**Split the data.** `stratify=` keeps the same fraud ratio in train and test.

In [ ]:
#@title 4. Train/test split { display-mode: "form" }
from sklearn.model_selection import train_test_split

# test_size=0.2 → 20% of rows go to test; random_state=42 → same split every run
# stratify=y_all → train and test keep the same ~0.17% fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all,
)
# y_train.mean() = fraud fraction inside the training set (labels are 0/1)
print(f"Train: {X_train.shape[0]:,}  ({y_train.mean()*100:.3f}% fraud)")
print(f"Test:  {X_test.shape[0]:,}  ({y_test.mean()*100:.3f}% fraud)")

## 5 · Logistic Regression

Classic fast model → outputs a fraud *probability*. `StandardScaler` rescales `Amount`/`Time`.

**Train logistic regression.**

In [ ]:
#@title 5a. Train logistic regression { display-mode: "form" }
need("X_train")   # stop with a clear message if the split hasn't run yet
from sklearn.linear_model import LogisticRegression   # linear model → outputs P(fraud)
from sklearn.preprocessing import StandardScaler       # rescales features to mean 0, std 1
from sklearn.pipeline import make_pipeline             # chains scaler → model in one object

# Pipeline = scaler + model chained together
# (the scaler is fitted on the training data only, inside .fit())
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
lr.fit(X_train, y_train)        # learn weights from training data only
pred_lr = lr.predict(X_test)    # predict labels (0/1) on unseen test data
print("Trained.")

**Precision & recall.** Precision = share of alarms that are real; recall = share of frauds caught.

In [ ]:
#@title 5b. Precision & recall { display-mode: "form" }
need("pred_lr")   # ensure predictions exist
from sklearn.metrics import precision_score, recall_score, classification_report

# zero_division=0 avoids a warning if the model predicts zero frauds
prec = precision_score(y_test, pred_lr, zero_division=0)   # of predicted frauds, how many real?
rec  = recall_score(y_test, pred_lr)                       # of real frauds, how many caught?
print(f"Logistic Regression — precision: {prec:.3f}   recall: {rec:.3f}")
print()
# per-class report: precision/recall/F1 for both legit and fraud
print(classification_report(y_test, pred_lr, target_names=["legit","fraud"], digits=3))

**Confusion matrix** — top-right = false alarms, bottom-left = missed frauds.

In [ ]:
#@title 5c. Confusion matrix { display-mode: "form" }
need("pred_lr", "rec")   # needs predictions and the recall value for the title
plot_cm(y_test, pred_lr, title=f"Logistic Regression — recall {rec:.2f}")

<details><summary>💡 Checkpoint — read the matrix</summary>

Top-right = false alarms (legit flagged fraud). Bottom-left = **missed frauds**.
Missed fraud ~$500 vs false alarm ~$5 → bottom-left hurts most → we want **recall** up,
even if precision drops.</details>

## 6 · Trees, forests & class weights

**Decision tree** = yes/no questions on features. **Random forest** = hundreds of trees averaged.

**Decision tree** — try different `max_depth` values and watch train vs test accuracy.

In [ ]:
#@title 6a. Decision tree — overfitting knob { display-mode: "form" }
need("X_train")
from sklearn.tree import DecisionTreeClassifier   # tree of yes/no splits on features

max_depth = 10          # 🌳 KNOB — try 2, 5, 10, None and re-run

tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
tree.fit(X_train, y_train)   # learn the splits from training data only

# .score() = accuracy on that data; comparing train vs test exposes overfitting
print(f"max_depth = {max_depth}")
print(f"Train accuracy: {tree.score(X_train, y_train)*100:.1f}%")
print(f"Test  accuracy: {tree.score(X_test,  y_test )*100:.1f}%")
print("(train ≫ test → memorized the data = overfitting)")

Fraud is 0.17% of data — the model can score well while ignoring it.
**`class_weight="balanced"`** makes fraud cases count more during training.

**Train the forest** — `class_weight="balanced"` makes fraud count ~578× more.

In [ ]:
#@title 6b. Random forest { display-mode: "form" }
need("X_train")
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,         # number of trees in the forest
    max_depth=12,            # max depth per tree — caps memorization
    random_state=42,         # fixed seed → reproducible results
    class_weight="balanced", # fraud cases get ~578× weight in training
    n_jobs=-1,               # use all CPU cores
)
rf.fit(X_train, y_train)   # each tree learns on a random subset of the data
print("Forest trained WITH class_weight='balanced'.")

**Forest scores** — precision & recall on the test set.

In [ ]:
#@title 6c. Forest scores { display-mode: "form" }
need("rf")   # needs the trained forest
pred_rf = rf.predict(X_test)                    # predict 0/1 on the test set
prec_rf = precision_score(y_test, pred_rf, zero_division=0)
rec_rf  = recall_score(y_test, pred_rf)
print(f"Random Forest — precision: {prec_rf:.3f}   recall: {rec_rf:.3f}")

**Forest confusion matrix.**

In [ ]:
#@title 6d. Forest confusion matrix { display-mode: "form" }
need("pred_rf")   # needs the forest's predictions
plot_cm(y_test, pred_rf, title="Random Forest (class_weight='balanced')")

**All models side by side** — accuracy vs precision vs recall.

In [ ]:
#@title 6e. All models compared { display-mode: "form" }
need("pred_lr", "pred_rf", "dummy", "X_test")   # needs all 3 models' predictions
from sklearn.metrics import accuracy_score

# re-predict the dummy on test data so all models are judged on the same rows
models = {"Always-legit": dummy.predict(X_test),
          "Logistic Reg.": pred_lr, "Random Forest": pred_rf}
# build one row per model with its 3 scores
rows = [{"Model": k,
         "Accuracy": accuracy_score(y_test, p),
         "Precision": precision_score(y_test, p, zero_division=0),
         "Recall": recall_score(y_test, p)}
        for k, p in models.items()]
table = pd.DataFrame(rows).set_index("Model")   # model names become row labels
# green highlight = best recall across models
display(table.style.format({"Accuracy":"{:.3f}","Precision":"{:.3f}","Recall":"{:.3f}"})
              .highlight_max(subset=["Recall"], color=ACCENT2))

<details><summary>💡 What to look for</summary>

Baseline: best accuracy, worst recall (0%). Forest+weights catches far more fraud than
plain logistic regression — at the cost of more false alarms. That trade-off is tuned next.</details>

## 7 · The cost game

Model outputs a fraud **probability**; we pick a **threshold**. Missed fraud = **$500**, false alarm = **$5**.

**Find the cheapest threshold** by trying many values and summing the costs.

In [ ]:
#@title 7a. Probabilities + best threshold { display-mode: "form" }
need("rf")   # needs the trained forest
COST_FRAUD_MISSED, COST_FALSE_ALARM = 500, 5   # dollars per mistake type
# predict_proba returns P(legit), P(fraud) per row → [:,1] keeps P(fraud) only
proba = rf.predict_proba(X_test)[:, 1]
yv = np.asarray(y_test)                        # numpy array for fast boolean math

# Try 197 thresholds from 0.01 to 0.99 and total the cost for each
#   (yv==1)&(proba<t) = real frauds we'd miss → $500 each
#   (yv==0)&(proba>=t) = legit flagged as fraud → $5 each
grid = np.linspace(0.01, 0.99, 197)
costs = [(((yv==1)&((proba>=t).astype(int)==0)).sum()*COST_FRAUD_MISSED +  # missed frauds
          ((yv==0)&((proba>=t).astype(int)==1)).sum()*COST_FALSE_ALARM)  # false alarms
         for t in grid]
best_t = grid[int(np.argmin(costs))]           # argmin → index of cheapest threshold
print(f"Cheapest threshold by sweep: {best_t:.2f} → ${min(costs):,}")
print("Now drag the slider — does 0.5 win?")

**Drag the slider** — does threshold 0.5 win?

In [ ]:
#@title 7b. Threshold slider { display-mode: "form" }
need("proba")   # needs the fraud probabilities from the previous cell
import ipywidgets as widgets

def total_cost(threshold):
    p = (proba >= threshold).astype(int)   # flag as fraud if P(fraud) >= threshold
    missed = int(((yv == 1) & (p == 0)).sum())   # real frauds we let through
    false  = int(((yv == 0) & (p == 1)).sum())   # legit flagged as fraud
    return (f"threshold = {threshold:.2f}\n"
            f"  missed frauds: {missed}  →  ${missed*COST_FRAUD_MISSED:,}\n"
            f"  false alarms:  {false}  →  ${false*COST_FALSE_ALARM:,}\n"
            f"  TOTAL COST:    ${missed*COST_FRAUD_MISSED + false*COST_FALSE_ALARM:,}")

out = widgets.Output()   # box that displays the cost report
# recompute the report every time the slider moves
slider = widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01,
                             description="threshold", style={"description_width":"initial"})
slider.observe(lambda ch: (out.clear_output(wait=True),
                           out.append_stdout(total_cost(ch["new"]))), names="value")
with out:
    print(total_cost(0.5))
display(widgets.VBox([slider, out]))

<details><summary>💡 Why isn't the best threshold 0.5?</summary>

0.5 ignores costs. A missed fraud costs 100× a false alarm → the cheapest threshold is
usually *lower* (~0.2): investigate many legit transactions to catch one more fraud.</details>

## 8 · Takeaways

- Fraud is rare → **accuracy lies**; use precision & recall.
- **Stratified split** + **class weights** make the model care about the rare class.
- Pick the **threshold by cost**, not habit.

🎉 Lab 1 complete.